# 20 — Real-data spatiotemporal tiny overfit

Prove that complete four-frame TorNet sequences flow from real NetCDF files through explicit sweep/variable axes, physical coordinates, the sweep-aware encoder, causal ConvGRU, loss, and backward pass. This is a bounded training-pipeline test, not a model-selection experiment.


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.8"
YEAR = 2013
VARIABLES = (
    "DBZ",
    "KDP",
    "RHOHV",
    "VEL",
    "WIDTH",
    "ZDR",
)
POSITIVE_FILE_COUNT = 12
WARNING_FILE_COUNT = 6
NULL_FILE_COUNT = 6
BATCH_SIZE = 4
MAX_EPOCHS = 200
LEARNING_RATE = 3e-3
TARGET_LOSS = 0.02
SEED = 20260923

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / f"tornet_detection-{PACKAGE_VERSION}-py3-none-any.whl"
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
NORMALIZATION_PATH = (
    BACKUP_ROOT
    / "experiments"
    / "all_year_all6_baseline_v1"
    / "normalization.json"
)
DRIVE_ARCHIVE = BACKUP_ROOT / f"tornet_{YEAR}.tar.gz"
LOCAL_ARCHIVE = Path(f"/content/tornet_{YEAR}.tar.gz")
EXTRACTION_ROOT = Path(
    "/content/tornet_spatiotemporal_tiny_overfit"
)

for path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    NORMALIZATION_PATH,
    DRIVE_ARCHIVE,
):
    if not path.exists():
        raise FileNotFoundError(path)

print("wheel:", PACKAGE_PATH)
print("archive:", DRIVE_ARCHIVE)
print("normalization:", NORMALIZATION_PATH)


wheel: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.8-py3-none-any.whl
archive: /content/drive/MyDrive/TorNet_Backup/tornet_2013.tar.gz
normalization: /content/drive/MyDrive/TorNet_Backup/experiments/all_year_all6_baseline_v1/normalization.json


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
        "scikit-learn>=1.5",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.8-py3-none-any.whl'], returncode=0)

In [4]:
import json
import random
import shutil
import tarfile
import time

import numpy as np
import torch
from sklearn.metrics import average_precision_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

import tornado_detection
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
    read_spatiotemporal_netcdf_file,
)
from tornado_detection.models import (
    SpatiotemporalTornadoDetector,
)

if tornado_detection.__version__ != PACKAGE_VERSION:
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )
if not torch.cuda.is_available():
    raise RuntimeError(
        "Select an A100 GPU runtime and run all cells"
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda")
normalization = json.loads(
    NORMALIZATION_PATH.read_text()
)

if normalization.get("variables") != list(VARIABLES):
    raise AssertionError(
        "Normalization variable provenance mismatch"
    )

means = np.asarray(
    normalization["means"],
    dtype=np.float32,
).reshape(len(VARIABLES), 2).T
stds = np.asarray(
    normalization["standard_deviations"],
    dtype=np.float32,
).reshape(len(VARIABLES), 2).T

assert means.shape == (2, 6)
assert stds.shape == (2, 6)
assert np.isfinite(means).all()
assert np.isfinite(stds).all()
assert (stds > 0).all()

canonical = load_canonical_frame_index(
    MANIFESTS_ROOT
)
assigned = assign_model_splits(
    canonical,
    validation_fraction=0.20,
    seed=20260913,
)
rows = assigned.loc[
    assigned["year"].eq(YEAR)
    & assigned["model_split"].eq("train")
].copy()

file_summary = (
    rows.groupby("archive_member", sort=True)
    .agg(
        category=("category", "first"),
        positive_frames=("frame_label", "sum"),
        frame_count=("frame_label", "size"),
    )
)

if not file_summary["frame_count"].eq(4).all():
    raise AssertionError(
        "Every selected file must contain four frames"
    )

positive_members = list(
    file_summary.loc[
        file_summary["positive_frames"].gt(0)
    ].index[:POSITIVE_FILE_COUNT]
)
warning_members = list(
    file_summary.loc[
        file_summary["positive_frames"].eq(0)
        & file_summary["category"].eq("WRN")
    ].index[:WARNING_FILE_COUNT]
)
null_members = list(
    file_summary.loc[
        file_summary["positive_frames"].eq(0)
        & file_summary["category"].eq("NUL")
    ].index[:NULL_FILE_COUNT]
)
selected_members = (
    positive_members
    + warning_members
    + null_members
)

if len(selected_members) != 24:
    raise AssertionError(
        f"Expected 24 files; selected {len(selected_members)}"
    )

records = []

for member in selected_members:
    member_rows = (
        rows.loc[rows["archive_member"].eq(member)]
        .sort_values("frame_index")
    )
    records.append(
        (
            member,
            member_rows["frame_label"]
            .astype(np.uint8)
            .to_numpy(),
        )
    )

print("GPU:", torch.cuda.get_device_name(0))
print("selected files:", len(records))
print(
    "selected positive frames:",
    int(sum(labels.sum() for _, labels in records)),
)
print("categories: 12 TOR-positive, 6 WRN, 6 NUL")


GPU: NVIDIA A100-SXM4-40GB
selected files: 24
selected positive frames: 20
categories: 12 TOR-positive, 6 WRN, 6 NUL


In [5]:
if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)
EXTRACTION_ROOT.mkdir(parents=True, exist_ok=False)

if LOCAL_ARCHIVE.exists():
    LOCAL_ARCHIVE.unlink()

copy_started = time.perf_counter()
shutil.copyfile(DRIVE_ARCHIVE, LOCAL_ARCHIVE)
print(
    "archive copy seconds:",
    round(time.perf_counter() - copy_started, 3),
)

required = set(selected_members)
extracted = set()

with tarfile.open(LOCAL_ARCHIVE, mode="r:gz") as archive:
    for member in archive:
        if not member.isfile() or member.name not in required:
            continue

        destination = EXTRACTION_ROOT / member.name
        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )
        source_file = archive.extractfile(member)

        if source_file is None:
            raise RuntimeError(member.name)

        with source_file, destination.open("wb") as output_file:
            shutil.copyfileobj(
                source_file,
                output_file,
                length=1024 * 1024,
            )

        extracted.add(member.name)

missing = required - extracted

if missing:
    raise RuntimeError(
        f"Missing archive members: {sorted(missing)}"
    )

LOCAL_ARCHIVE.unlink()

print("extracted files:", len(extracted))
print(
    "extracted MiB:",
    round(
        sum(
            path.stat().st_size
            for path in EXTRACTION_ROOT.rglob("*.nc")
        )
        / 1024**2,
        3,
    ),
)


archive copy seconds: 35.436
extracted files: 24
extracted MiB: 13.668


In [6]:
sample = read_spatiotemporal_netcdf_file(
    EXTRACTION_ROOT / records[0][0],
    variables=VARIABLES,
)

assert sample.values.shape == (4, 2, 6, 120, 240)
assert sample.finite_mask.shape == sample.values.shape
assert sample.range_folded_mask.shape == (4, 2, 120, 240)
assert sample.coordinates.shape == (2, 5, 120, 240)
assert sample.labels.shape == (4,)
assert sample.variables == VARIABLES
np.testing.assert_array_equal(
    sample.labels,
    records[0][1],
)

print("values:", sample.values.shape, sample.values.dtype)
print("finite mask:", sample.finite_mask.shape)
print("range-folded mask:", sample.range_folded_mask.shape)
print("coordinates:", sample.coordinates.shape)
print("coordinate names:", sample.coordinate_names)
print("labels:", sample.labels.tolist())
print(
    "time deltas seconds:",
    np.diff(sample.times_unix_seconds).tolist(),
)
print(
    "finite fraction:",
    round(float(sample.finite_mask.mean()), 6),
)
print(
    "sweep elevations:",
    sample.coordinates[:, 4, 0, 0].tolist(),
)


values: (4, 2, 6, 120, 240) float32
finite mask: (4, 2, 6, 120, 240)
range-folded mask: (4, 2, 120, 240)
coordinates: (2, 5, 120, 240)
coordinate names: ('range_100km', 'inverse_range_100km', 'sin_azimuth', 'cos_azimuth', 'elevation_degrees')
labels: [0, 0, 0, 1]
time deltas seconds: [300, 300, 300]
finite fraction: 0.217803
sweep elevations: [0.5, 0.8999999761581421]


In [7]:
class SequenceDataset(Dataset):
    def __init__(self, file_records, root, channel_means, channel_stds):
        self.records = file_records
        self.root = root
        self.means = channel_means.reshape(1, 2, 6, 1, 1)
        self.stds = channel_stds.reshape(1, 2, 6, 1, 1)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        member, expected_labels = self.records[index]
        sequence = read_spatiotemporal_netcdf_file(
            self.root / member,
            variables=VARIABLES,
        )
        np.testing.assert_array_equal(
            sequence.labels,
            expected_labels,
        )

        normalized = (
            (sequence.values - self.means)
            / self.stds
        ).astype(np.float32, copy=False)

        return (
            torch.from_numpy(normalized),
            torch.from_numpy(sequence.finite_mask),
            torch.from_numpy(sequence.range_folded_mask),
            torch.from_numpy(sequence.coordinates),
            torch.from_numpy(
                sequence.labels.astype(np.float32)
            ),
        )


dataset = SequenceDataset(
    records,
    EXTRACTION_ROOT,
    means,
    stds,
)
loader_generator = torch.Generator().manual_seed(SEED)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

model = SpatiotemporalTornadoDetector().to(device)
all_labels = np.concatenate(
    [labels for _, labels in records]
).astype(np.float32)
positive_count = float(all_labels.sum())
negative_count = float(len(all_labels) - positive_count)
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        negative_count / positive_count,
        device=device,
    )
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

batch = next(iter(loader))

with torch.no_grad():
    smoke_logits, smoke_maps = model.forward_with_maps(
        batch[0].to(device),
        batch[1].to(device),
        batch[2].to(device),
        batch[3].to(device),
    )

assert smoke_logits.shape == (BATCH_SIZE, 4)
assert smoke_maps.shape == (BATCH_SIZE, 4, 15, 30)
assert torch.isfinite(smoke_logits).all()
assert torch.isfinite(smoke_maps).all()

print(
    "parameters:",
    f"{sum(parameter.numel() for parameter in model.parameters()):,}",
)
print("smoke logits:", tuple(smoke_logits.shape))
print("smoke maps:", tuple(smoke_maps.shape))
print("positive weight:", negative_count / positive_count)


parameters: 2,011,201
smoke logits: (4, 4)
smoke maps: (4, 4, 15, 30)
positive weight: 3.8


In [8]:
history = []
training_started = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    frame_count = 0

    for (
        values,
        finite_mask,
        range_folded_mask,
        coordinates,
        labels,
    ) in loader:
        values = values.to(device, non_blocking=True)
        finite_mask = finite_mask.to(device, non_blocking=True)
        range_folded_mask = range_folded_mask.to(
            device,
            non_blocking=True,
        )
        coordinates = coordinates.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(
            values,
            finite_mask,
            range_folded_mask,
            coordinates,
        )
        loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}"
            )

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        loss_sum += float(loss.detach()) * labels.numel()
        frame_count += labels.numel()

    epoch_loss = loss_sum / frame_count
    history.append(epoch_loss)

    if (
        epoch == 1
        or epoch % 10 == 0
        or epoch_loss <= TARGET_LOSS
    ):
        print(
            f"epoch={epoch:03d} "
            f"loss={epoch_loss:.6f}"
        )

    if epoch_loss <= TARGET_LOSS:
        break

print(
    "training seconds:",
    round(time.perf_counter() - training_started, 3),
)
print("epochs:", len(history))
print("initial loss:", history[0])
print("final loss:", history[-1])


epoch=001 loss=1.024256
epoch=010 loss=0.660512
epoch=020 loss=0.314661
epoch=030 loss=0.037789
epoch=040 loss=0.135473
epoch=048 loss=0.019865
training seconds: 42.976
epochs: 48
initial loss: 1.0242560307184856
final loss: 0.019865005277097225


In [9]:
evaluation_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

model.eval()
evaluation_labels = []
evaluation_probabilities = []

with torch.no_grad():
    for (
        values,
        finite_mask,
        range_folded_mask,
        coordinates,
        labels,
    ) in evaluation_loader:
        logits = model(
            values.to(device, non_blocking=True),
            finite_mask.to(device, non_blocking=True),
            range_folded_mask.to(
                device,
                non_blocking=True,
            ),
            coordinates.to(device, non_blocking=True),
        )
        evaluation_labels.extend(
            labels.numpy().reshape(-1).tolist()
        )
        evaluation_probabilities.extend(
            logits.sigmoid()
            .cpu()
            .numpy()
            .reshape(-1)
            .tolist()
        )

evaluation_labels = np.asarray(
    evaluation_labels,
    dtype=np.int64,
)
evaluation_probabilities = np.asarray(
    evaluation_probabilities,
    dtype=np.float64,
)
predictions = (
    evaluation_probabilities >= 0.5
).astype(np.int64)
accuracy = float(
    (predictions == evaluation_labels).mean()
)
pr_auc = float(
    average_precision_score(
        evaluation_labels,
        evaluation_probabilities,
    )
)

if history[-1] > TARGET_LOSS:
    raise AssertionError(
        f"Failed to reach target loss: {history[-1]:.6f}"
    )
if accuracy != 1.0:
    raise AssertionError(
        "Failed to memorize fixed sequences: "
        f"accuracy={accuracy}"
    )
if pr_auc < 0.999:
    raise AssertionError(
        f"Tiny-set PR-AUC is too low: {pr_auc}"
    )

print("tiny-set accuracy:", accuracy)
print("tiny-set PR-AUC:", pr_auc)
print(
    "minimum probability:",
    evaluation_probabilities.min(),
)
print(
    "maximum probability:",
    evaluation_probabilities.max(),
)
print("REAL-DATA SPATIOTEMPORAL OVERFIT: PASSED")


tiny-set accuracy: 1.0
tiny-set PR-AUC: 1.0
minimum probability: 0.0002664329076651484
maximum probability: 0.9999380111694336
REAL-DATA SPATIOTEMPORAL OVERFIT: PASSED


In [10]:
shutil.rmtree(EXTRACTION_ROOT)

assert not EXTRACTION_ROOT.exists()
assert not LOCAL_ARCHIVE.exists()

print("Removed all Colab-local staged data")


Removed all Colab-local staged data
